## Sobrevivendo ao ` Titanic `

O ` Titanic `, um dos navios mais famosos da história, naufragou em 1912, deixando um legado de histórias e aprendizados. Entre os dados coletados desse desastre, estão informações detalhadas sobre os passageiros, como classe do bilhete, idade, gênero e se sobreviveram ou não.

### O Desafio

1. **Treinar um modelo** – Utilize os dados do arquivo ` titanic_train.csv ` para prever a sobrevivência dos passageiros.
2. **Evolução** – Comece com um ` Perceptron ` simples (linear) e evolua para um **Perceptron Multicamadas (MLP)** para lidar com a complexidade dos dados.
3. **Gerar previsões** – Salve o resultado final em ` solution_sobrevivendo_ao_titanic.csv `.

Após implementar sua solução, você poderá ver as diversas formas de resolver o problema abaixo:

In [ ]:
# Escreva seu código aqui


### Soluções

Abaixo você encontrará diferentes formas de resolver o problema em células independentes e executáveis.

#### 1.1 Perceptron Simples - Utilizando Scikit-Learn

In [8]:
import pandas as pd
from sklearn.linear_model import Perceptron
from sklearn.preprocessing import StandardScaler

# 1. Carregar e Limpar
train = pd.read_csv('../arquivos-csv/titanic_train.csv')
test = pd.read_csv('../arquivos-csv/titanic_exercise.csv')

def preprocess(df):
    df['title'] = df['name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    title_mapping = {'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4}
    df['title_idx'] = df['title'].map(title_mapping).fillna(5)
    X = pd.DataFrame()
    X['pclass'] = df['pclass']
    X['is_male'] = df['sex'].map({'male': 1, 'female': 0})
    X['age'] = pd.to_numeric(df['age'], errors='coerce')
    X['family'] = df['sibsp'] + df['parch'] + 1
    X['fare'] = pd.to_numeric(df['fare'], errors='coerce')
    X['title'] = df['title_idx']
    X['age'] = X.groupby('title')['age'].transform(lambda x: x.fillna(x.median()))
    X = X.fillna(X.median())
    return X

X_train, y_train = preprocess(train), train['survived']
X_test = preprocess(test)

# 2. Normalizar e Treinar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
clf = Perceptron(random_state=42)
clf.fit(X_train_scaled, y_train)

# 3. Salvar Resposta
y_pred = clf.predict(scaler.transform(X_test))
pd.DataFrame({'id': test['id'], 'survived': y_pred}).to_csv('solution_perceptron_sklearn.csv', index=False)
print("CSV gerado: Perceptron (Scikit-Learn)")

CSV gerado: Perceptron (Scikit-Learn)


<>:10: SyntaxWarning: invalid escape sequence '\.'
<>:10: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_18356/4052151501.py:10: SyntaxWarning: invalid escape sequence '\.'
  df['title'] = df['name'].str.extract(' ([A-Za-z]+)\.', expand=False)


#### 1.2 Perceptron Simples - Utilizando Python Puro

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Carregar e Preparar
train = pd.read_csv('../arquivos-csv/titanic_train.csv')
test = pd.read_csv('../arquivos-csv/titanic_exercise.csv')

def preprocess(df):
    df['title'] = df['name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    title_mapping = {'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4}
    df['title_idx'] = df['title'].map(title_mapping).fillna(5)
    X = pd.DataFrame()
    X['pclass'] = df['pclass']
    X['is_male'] = df['sex'].map({'male': 1, 'female': 0})
    X['age'] = pd.to_numeric(df['age'], errors='coerce')
    X['family'] = df['sibsp'] + df['parch'] + 1
    X['fare'] = pd.to_numeric(df['fare'], errors='coerce')
    X['title'] = df['title_idx']
    X['age'] = X.groupby('title')['age'].transform(lambda x: x.fillna(x.median()))
    X = X.fillna(X.median())
    return X

X_train, y_train = preprocess(train), train['survived'].values
X_test = preprocess(test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Algoritmo Perceptron
w = np.zeros(X_train_scaled.shape[1] + 1)
lr, epochs = 0.01, 100
for _ in range(epochs):
    for i in range(len(X_train_scaled)):
        x = np.insert(X_train_scaled[i], 0, -1)
        y_hat = 1 if np.dot(w, x) >= 0 else 0
        w += lr * (y_train[i] - y_hat) * x

# 3. Previsão e CSV
preds = []
for x_row in X_test_scaled:
    preds.append(1 if np.dot(w, np.insert(x_row, 0, -1)) >= 0 else 0)

pd.DataFrame({'id': test['id'], 'survived': preds}).to_csv('solution_perceptron_puro.csv', index=False)
print("CSV gerado: Perceptron (Puro)")

<>:10: SyntaxWarning: invalid escape sequence '\.'
<>:10: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_18356/242907240.py:10: SyntaxWarning: invalid escape sequence '\.'
  df['title'] = df['name'].str.extract(' ([A-Za-z]+)\.', expand=False)


CSV gerado: Perceptron (Puro)


#### 2.1 MLP (Rede Neural) - Utilizando Scikit-Learn

In [7]:
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# 1. Carregar e Limpar
train = pd.read_csv('../arquivos-csv/titanic_train.csv')
test = pd.read_csv('../arquivos-csv/titanic_exercise.csv')

def preprocess(df):
    df['title'] = df['name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    title_mapping = {'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4}
    df['title_idx'] = df['title'].map(title_mapping).fillna(5)
    X = pd.DataFrame()
    X['pclass'] = df['pclass']
    X['is_male'] = df['sex'].map({'male': 1, 'female': 0})
    X['age'] = pd.to_numeric(df['age'], errors='coerce')
    X['family'] = df['sibsp'] + df['parch'] + 1
    X['fare'] = pd.to_numeric(df['fare'], errors='coerce')
    X['title'] = df['title_idx']
    X['age'] = X.groupby('title')['age'].transform(lambda x: x.fillna(x.median()))
    X = X.fillna(X.median())
    return X

X_train, y_train = preprocess(train), train['survived']
X_test = preprocess(test)

# 2. Normalizar e Treinar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=2000, alpha=0.01, random_state=42)
mlp.fit(X_train_scaled, y_train)

# 3. Salvar Resposta
y_pred = mlp.predict(scaler.transform(X_test))
pd.DataFrame({'id': test['id'], 'survived': y_pred}).to_csv('solution_mlp_sklearn.csv', index=False)
print("CSV gerado: MLP (Scikit-Learn)")

<>:10: SyntaxWarning: invalid escape sequence '\.'
<>:10: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_18356/1324916766.py:10: SyntaxWarning: invalid escape sequence '\.'
  df['title'] = df['name'].str.extract(' ([A-Za-z]+)\.', expand=False)


CSV gerado: MLP (Scikit-Learn)


#### 2.2 MLP (Rede Neural) - Utilizando Python Puro

In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Carregar e Preparar
train = pd.read_csv('../arquivos-csv/titanic_train.csv')
test = pd.read_csv('../arquivos-csv/titanic_exercise.csv')

def preprocess(df):
    df['title'] = df['name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    title_mapping = {'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4}
    df['title_idx'] = df['title'].map(title_mapping).fillna(5)
    X = pd.DataFrame()
    X['pclass'] = df['pclass']
    X['is_male'] = df['sex'].map({'male': 1, 'female': 0})
    X['age'] = pd.to_numeric(df['age'], errors='coerce')
    X['family'] = df['sibsp'] + df['parch'] + 1
    X['fare'] = pd.to_numeric(df['fare'], errors='coerce')
    X['title'] = df['title_idx']
    X['age'] = X.groupby('title')['age'].transform(lambda x: x.fillna(x.median()))
    X = X.fillna(X.median())
    return X

X_train, y_train = preprocess(train), train['survived']
X_test = preprocess(test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

class MLP:
    def __init__(self, d_in, d_hid, d_out):
        self.w1 = np.random.randn(d_in, d_hid)
        self.w2 = np.random.randn(d_hid, d_out)
        self.lr = 0.1
    def train(self, X, y, ep=2000):
        y_reshaped = y.values.reshape(-1,1)
        for _ in range(ep):
            h = 1 / (1 + np.exp(-np.dot(X, self.w1)))
            o = 1 / (1 + np.exp(-np.dot(h, self.w2)))
            do = (y_reshaped - o) * (o * (1 - o))
            dh = do.dot(self.w2.T) * (h * (1 - h))
            self.w2 += h.T.dot(do) * self.lr
            self.w1 += X.T.dot(dh) * self.lr
    def pred(self, X):
        h = 1 / (1 + np.exp(-np.dot(X, self.w1)))
        o = 1 / (1 + np.exp(-np.dot(h, self.w2)))
        return [1 if i > 0.5 else 0 for i in o]

# Treinar e Salvar
net = MLP(X_train_scaled.shape[1], 16, 1)
net.train(X_train_scaled, y_train)
y_res = net.pred(X_test_scaled)
pd.DataFrame({'id': test['id'], 'survived': y_res}).to_csv('solution_mlp_puro.csv', index=False)
print("CSV gerado: MLP (Puro)")

<>:10: SyntaxWarning: invalid escape sequence '\.'
<>:10: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_18356/1498111371.py:10: SyntaxWarning: invalid escape sequence '\.'
  df['title'] = df['name'].str.extract(' ([A-Za-z]+)\.', expand=False)


CSV gerado: MLP (Puro)
